# Run 001 — Qwen2-VL-7B-Instruct, 8-bit QLoRA, 10k samples| | ||---|---|| **run_id** | `run001_qwen2vl_8bit_10k` || **Model** | `Qwen/Qwen2-VL-7B-Instruct` || **Quantisation** | 8-bit (`bitsandbytes` LLM.int8) — flip `quantization.bits` to `4` if this OOMs || **Adapter** | LoRA `r=16`, `alpha=32`, `dropout=0.05` on `q_proj,k_proj,v_proj,o_proj` of the **language model only** (vision tower frozen) || **Visual tokens** | `max_pixels = 256 × 28 × 28` (~256 tokens/image) — *the* memory knob || **Batch** | 1 × 8 accumulation = effective 8 || **Schedule** | 3 epochs, cosine, `warmup_ratio=0.03`, `lr=2e-4`, `paged_adamw_8bit`, fp16 || **Train / Eval** | 10,000 stratified rows / **frozen 5,000-row eval split** |### What this run testsWhether an 8-bit QLoRA fine-tune of a 7B VLM on 10k samples learns the**exact output format** the metric demands, and how much of the remaining gappost-processing closes. F1 is reported **both raw and post-processed** so thevalue of normalisation is visible rather than assumed.### Expected wall-clock (T4)~7–9 hours total: ~5.5–7.5 h training (3,750 optimiser steps) + ~35–50 mingeneration over 5,000 eval rows + ~20 min image download. Inside Kaggle's12-hour limit, but not by a wide margin — **use T4 ×2, not P100**.### Non-negotiables this notebook respects* The 5k eval split is **frozen and committed**; it is verified, never regenerated.* No logic lives in this notebook — every cell calls into `src/amlc24/`.

## 1. Setup — locate the repo

In [ ]:
# Locate the repo and put its `src/` on sys.path.## Three layouts are supported, in priority order:#   1. the repo uploaded as a Kaggle Dataset  -> /kaggle/input/<slug>/.../src#   2. the repo git-cloned into the session   -> /kaggle/working/Amazon-ML-24/src#   3. running locally from the repo itself## Nothing below hardcodes a dataset slug: paths.py does the environment# detection, and this cell only has to find the package.import sys, glob, osfrom pathlib import Pathdef find_src():    candidates = []    for root in sorted(glob.glob("/kaggle/input/*/")):        candidates += [Path(root) / "src", *[Path(p) for p in glob.glob(root + "*/src")]]    candidates += [Path(p) for p in sorted(glob.glob("/kaggle/working/*/src"))]    candidates += [Path.cwd() / "src", Path.cwd().parent / "src"]    for c in candidates:        if (c / "amlc24" / "__init__.py").exists():            return c.resolve()    return NoneSRC = find_src()if SRC is None:    # Not mounted anywhere: clone it (Kaggle notebooks have internet enabled).    !git clone --depth 1 https://github.com/MurtuzaShaikh26/Amazon-ML-24.git /kaggle/working/Amazon-ML-24    SRC = Path("/kaggle/working/Amazon-ML-24/src")sys.path.insert(0, str(SRC))print("src:", SRC)import amlc24from amlc24.logging_utils import setup_loggingfrom amlc24.paths import describesetup_logging()print("amlc24", amlc24.__version__)for k, v in describe().items():    print(f"  {k:20s} {v}")

## 2. Guarded installsKaggle's base image already carries most of this. We only install what ismissing or too old, then print the resolved versions.

In [ ]:
import importlib, subprocess, sysREQUIRED = {    "transformers": "4.45.0",    "peft": "0.13.0",    "bitsandbytes": "0.44.0",    "accelerate": "0.34.0",    "trl": "0.11.0",    "qwen_vl_utils": None,   # pip name: qwen-vl-utils}PIP_NAME = {"qwen_vl_utils": "qwen-vl-utils"}def version_of(mod):    try:        return importlib.metadata.version(PIP_NAME.get(mod, mod))    except Exception:        return Nonemissing = []for mod, min_version in REQUIRED.items():    have = version_of(mod)    if have is None:        missing.append(PIP_NAME.get(mod, mod))    elif min_version and tuple(map(int, have.split(".")[:3])) < tuple(map(int, min_version.split("."))):        missing.append(f"{PIP_NAME.get(mod, mod)}>={min_version}")if missing:    print("Installing:", missing)    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", *missing], check=False)else:    print("All requirements already satisfied.")

In [ ]:
import importlibfor mod in ["torch", "transformers", "peft", "bitsandbytes", "accelerate", "trl"]:    try:        m = importlib.import_module(mod)        print(f"{mod:16s} {getattr(m, '__version__', '?')}")    except ImportError as exc:        print(f"{mod:16s} MISSING ({exc})")

## 3. GPU check**T4 is required.** T4 is Turing (SM 7.5): no bf16 and no flash-attention-2,which is why the config uses fp16 + `sdpa`. A P100 will run but is slower andrisks exceeding the 12-hour session limit.

In [ ]:
from amlc24.models.qwen2vl import gpu_reportreport = gpu_report()for k, v in report.items():    print(f"  {k:22s} {v}")if not report.get("cuda"):    raise RuntimeError("No GPU. Set Accelerator to 'GPU T4 x2' in notebook settings.")if report.get("is_p100"):    print("\n" + "!" * 76)    print("!!  WARNING: P100 detected, not T4.")    print("!!  P100 is materially slower for this fp16 workload and this run may")    print("!!  exceed the 12-hour Kaggle session limit.")    print("!!  Switch the accelerator to 'GPU T4 x2' and restart the session.")    print("!" * 76)elif report.get("is_t4"):    print("\nT4 confirmed — fp16 + sdpa, as configured.")if report.get("total_vram_gb", 0) < 15:    print(f"\nWARNING: only {report.get('total_vram_gb')} GB VRAM visible; "          "consider quantization.bits = 4.")

## 4. ConfigLoaded from `configs/run001_qwen2vl_8bit_10k.yaml`, which `extends: base.yaml`.The `config_hash` is recorded on the leaderboard row so any score traces back toexact settings.

In [ ]:
from amlc24.config import config_hash, load_configCONFIG = "run001_qwen2vl_8bit_10k"cfg = load_config(CONFIG)print(f"run_id      {cfg.run_id}")print(f"hash        {config_hash(cfg)}")print(f"description {cfg.description}")print(f"\nmodel       {cfg.model.id}  ({cfg.quantization.bits}-bit, attn={cfg.model.attn_implementation})")print(f"visual toks {cfg.processor.max_pixels_tokens} max  ({cfg.processor.max_pixels_tokens * 28 * 28} px)")print(f"lora        r={cfg.lora.r} alpha={cfg.lora.alpha} targets={list(cfg.lora.target_modules)}")print(f"train       {cfg.train.num_train_epochs} epochs, bs={cfg.train.per_device_train_batch_size}"      f" x accum={cfg.train.gradient_accumulation_steps}, lr={cfg.train.learning_rate}")print(f"data        {cfg.data.train_size} train / {cfg.data.eval_size} eval (FROZEN)")

## 5. ImagesThe pipeline downloads only the ~15k images the frozen split references, resizedso the longest side is 448px (matching the `max_pixels` cap) and saved as JPEGq90. Existing files are skipped, so this is resumable and cheap to re-run.**If you have already uploaded the resized images as a private Kaggle Dataset**,`paths.py` finds it automatically and this step becomes a no-op — which is therecommended workflow, since Kaggle sessions are ephemeral and re-downloading15k images wastes ~20 minutes of the 12-hour budget every run.

In [ ]:
from amlc24.paths import IMAGE_DIR, on_kaggleprint("Image dir:", IMAGE_DIR)print("Exists:   ", IMAGE_DIR.exists())if IMAGE_DIR.exists():    n = sum(1 for _ in IMAGE_DIR.glob("*.jpg"))    print(f"Cached:    {n:,} jpg files")    if n > 10000:        print("-> Pre-uploaded image dataset detected; download will be a no-op.")    else:        print("-> Will download the missing images (~20 min for a cold start).")else:    print("-> No image cache; the pipeline will download into the working dir.")

## 6. Split creation / verificationThe 5,000-row eval split is **frozen**: it is loaded from`results/splits/split_seed42.json`, regenerated, and compared. Any disagreementraises `SplitMismatch` rather than silently overwriting — every committed scoredepends on those exact rows.The table below is the stratification check: `entity_name` proportions mustmatch across the full file, the eval split, and the training subset to within1 percentage point.

In [ ]:
from amlc24.data.load import load_trainfrom amlc24.data.splits import get_or_create_splittrain_all = load_train()split = get_or_create_split(    train_all,    seed=int(cfg.seed),    eval_size=int(cfg.data.eval_size),    train_size=int(cfg.data.train_size),    path=cfg.data.split_file,)print(f"eval_5k       {len(split['eval_5k']):,} rows  (FROZEN, immutable)")print(f"train_subset  {len(split.get('train_subset', [])):,} rows")print(f"pool          {len(split.get('pool', [])):,} rows")print(f"overlap       {len(set(split['eval_5k']) & set(split.get('train_subset', [])))}")print(f"strata        {split['stratify']['n_strata']}  key = {split['stratify']['key']}")table = split.get("proportion_table")if table is not None:    import pandas as pd    display(pd.DataFrame(table) if not hasattr(table, "columns") else table)

## 7. TrainOne call. It downloads images, builds the datasets with completion-only labelmasking, loads the quantised model, attaches LoRA, trains for 3 epochs,generates over the frozen eval 5k, scores raw **and** post-processed, and writesevery artefact under `results/runs/run001_qwen2vl_8bit_10k/`.Expect **~6–8 hours**. Loss is logged every 25 steps and eval loss every epoch.

### Class weightsThe EDA measured a **31.5x** imbalance across `entity_name` (`item_weight`38.95% vs `maximum_weight_recommendation` 1.24%). Unweighted, ~72% of thegradient comes from weight-and-dimension rows.`train.class_weights` applies a per-sample loss multiplier. `sqrt_inverse`(the default) caps the spread near 5.6x; full `inverse` would hand the rarestclass a 31x multiplier, which at batch size 1 makes gradient magnitude swinghard between micro-batches -- an fp16 loss-spike risk.

In [ ]:
from amlc24.data.load import load_split_framesfrom amlc24.train.weighting import weights_from_config_eval_df, _train_df = load_split_frames(split, train_all)class_weights = weights_from_config(    _train_df["entity_name"].tolist(), cfg.get("train", {}).get("class_weights", {}))if class_weights:    import pandas as pd    counts = _train_df["entity_name"].value_counts()    display(pd.DataFrame({        "entity_name": list(class_weights),        "n_train": [int(counts.get(e, 0)) for e in class_weights],        "loss_weight": [round(w, 3) for w in class_weights.values()],    }).sort_values("n_train", ascending=False).reset_index(drop=True))    print(f"spread (max/min) = {max(class_weights.values())/min(class_weights.values()):.2f}x")else:    print("Class weighting disabled -- using the model's own unweighted loss.")

In [ ]:
from amlc24.pipeline.run_finetune import run_finetuneresult = run_finetune(CONFIG)metrics = result["metrics"]print("\nDone.")print(f"  F1 raw            {metrics['raw']['f1']:.4f}")print(f"  F1 post-processed {metrics['post']['f1']:.4f}")

## 8. Results

### 8.1 Loss curves

In [ ]:
from amlc24.train.trainer import plot_loss_curveshistory = result["history"]if len(history):    display(history.tail(10))    plot_loss_curves(history);else:    print("No loss history recorded.")

### 8.2 Overall F1 — raw vs post-processedThe delta is the measured value of `postprocess/`. If it is large, the modelknows the answer but not the format, and more normalisation rules pay off fasterthan more training.

In [ ]:
import pandas as pdraw, post = result["raw"], result["post"]summary = pd.DataFrame([    {"variant": "raw generation",  **{k: raw[k]  for k in ["f1", "precision", "recall", "tp", "fp", "fn", "tn"]}},    {"variant": "post-processed",  **{k: post[k] for k in ["f1", "precision", "recall", "tp", "fp", "fn", "tn"]}},])display(summary.style.format({"f1": "{:.4f}", "precision": "{:.4f}", "recall": "{:.4f}"}))print(f"Post-processing changed micro F1 by {post['f1'] - raw['f1']:+.4f}")print()print("MICRO vs MACRO F1")print(f"  micro (overall)      raw={raw['f1']:.4f}  post={post['f1']:.4f}")print(f"  macro (per-entity)   raw={result['macro_f1_raw']:.4f}  post={result['macro_f1_post']:.4f}")print()print("  Micro F1 is dominated by item_weight (38.95% of the data). Macro F1")print("  weights all eight entities equally, so it is the number that actually")print("  moves when class-weighted training helps a rare entity.")print("Post-processing rules fired:")for rule, count in sorted(metrics["postprocess_rules"].items(), key=lambda kv: -kv[1])[:15]:    print(f"  {rule:40s} {count:,}")

### 8.3 F1 by entity — the class-wise evaluationEntities differ a lot in how legible their value is in a product photo, and theoverall number hides that. `f1_delta_from_postprocess` shows which entitiespost-processing rescued.

In [ ]:
display(result["by_entity"].style.format({c: "{:.4f}" for c in ["f1", "precision", "recall", "f1_raw", "f1_delta_from_postprocess", "accuracy"]}))

### 8.4 F1 by ground-truth unitGrouping by the *true* unit separates unit-formatting failures from valuefailures — two different problems with two different fixes.

In [ ]:
display(result["by_unit"].head(30).style.format({c: "{:.4f}" for c in ["f1", "precision", "recall", "accuracy"]}))

### 8.5 F1 by `group_id` — the category-wise breakdownThere are 750 product categories with a long tail, so groups with fewer than 20eval rows are pooled. A category scoring far below the rest usually means aproduct type whose packaging the model cannot read — a data problem rather thana formatting one.

In [ ]:
by_group = result.get("by_group")if by_group is not None:    display(by_group.head(25).style.format(        {c: "{:.4f}" for c in ["f1", "precision", "recall", "accuracy"]}))else:    print("No group_id column in the predictions frame.")

### 8.6 Error analysis — where the next rules come fromThe most frequent `(predicted, actual)` mismatch pairs per entity. A repeatedpair with `same_number_diff_unit = True` is a missing alias and is free score;scattered numeric errors are a model-capacity problem instead.

In [ ]:
errors = result["errors"]recoverable = errors[errors["same_number_diff_unit"]]print(f"{len(errors)} distinct mismatch pairs; "      f"{len(recoverable)} differ only in the unit (recoverable by post-processing)")display(errors.head(40))

In [ ]:
# Sample of raw vs post-processed predictions, for a qualitative read.preds = result["predictions"]display(preds[preds["y_true"] != preds["y_pred_post"]].head(25))

### 8.7 Leaderboard

In [ ]:
from amlc24.results.tracker import read_leaderboarddisplay(read_leaderboard())

## 9. Package the run for downloadZips `results/runs/run001_qwen2vl_8bit_10k/` (metrics, predictions, breakdowns,logs — excluding the adapter binaries). Commit the extracted contents so theleaderboard and metrics stay in the repo.The LoRA adapter itself is saved separately under `checkpoints/adapter_best/`(~40 MB); download it from the Kaggle output pane if you want to reuse it.

In [ ]:
tracker = result["tracker"]zip_path = tracker.zip_artifacts()print("Run artefacts:", tracker.dir)for p in sorted(tracker.dir.rglob("*")):    if p.is_file():        print(f"  {p.relative_to(tracker.dir)}  ({p.stat().st_size / 1024:.1f} KB)")print("\nZipped to:", zip_path)

## 10. Observations<!-- Fill this in after the run. Record what you learned in NOTES.md too. -->